# RL-Chess-Engine: does a history-aware search plus contempt break the draw cycle?

Two findings from 2026-11-16/17 motivate this run.

1. **A search bug.** `ChessGame.clone()` copied the board with `stack=False`, so inside MCTS a move
   that completed a threefold with positions from *earlier in the game* looked like any other move.
   The search could not see the repetition it was walking into. Clones now keep the reversible
   history (`--legacy-search-history` restores the old behaviour).
2. **The material draw target did not help** (`draw_fixes.ipynb`, report §5.9): change vs start
   -93 [-122, -63], the same as the baseline. The value head learned a little material
   (+0.08 up-minus-down a queen) but self-play stayed 87-95 % non-decisive, mostly repetitions.

Three arms, baseline settings otherwise, init seeds 1-3 (the same starting networks):

| arm | change |
|---|---|
| `history` | search sees the game's history (the bug fix alone) |
| `contempt` | + self-play contempt 0.2: a drawn leaf is worth -0.2 to the side to move at the root |
| `contempt-material` | + contempt 0.2 and the material draw target |

Evaluation always searches with contempt 0. The arms run side by side (one CPU core each,
two T4s), each in its own clone so logs and PGN folders cannot collide. **Every finished seed is
zipped into `/kaggle/working` at once**, so the 12 h cap cannot lose finished work.

In [ ]:
ARMS = {
    "history": "",
    "contempt": "--contempt 0.2",
    "contempt-material": "--contempt 0.2 --draw-target material",
}
GPU = {"history": "0", "contempt": "1", "contempt-material": "0"}
SEEDS = [1, 2, 3]
REPO = "https://github.com/Danny-397/RL-Chess-Engine.git"


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc
!pip install -q python-chess onnx onnxscript
import os, subprocess, time
for arm in ARMS:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, f"/tmp/rl-{arm}"], check=True)
print(subprocess.run(["git", "-C", "/tmp/rl-history", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)


## Launch the three lanes

In [ ]:
os.makedirs("/kaggle/working/logs", exist_ok=True)
procs = {}
for arm, flags in ARMS.items():
    steps = []
    for s in SEEDS:
        steps.append(f"python run_experiment.py --device cuda --skip-scaled --skip-assisted "
                     f"--seed {s} --tag {arm} {flags}")
        steps.append(f"(cd experiment_out && zip -qr /kaggle/working/{arm}-seed{s}.zip "
                     f"seed{s}-{arm} -x '*/checkpoints/*')")
    cks = " ".join(f"--checkpoint experiment_out/seed{s}-{arm}/baseline.pt" for s in SEEDS)
    steps.append(f"python tools/value_probe.py --no-untrained {cks} "
                 f"--out /kaggle/working/value_probe_{arm}.json")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=GPU[arm], OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
    log = open(f"/kaggle/working/logs/{arm}.log", "w")
    procs[arm] = subprocess.Popen(" && ".join(steps), shell=True, cwd=f"/tmp/rl-{arm}",
                                  env=env, stdout=log, stderr=subprocess.STDOUT)
    print("started", arm)


In [ ]:
t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(600)
    done = sorted(f for f in os.listdir("/kaggle/working") if f.endswith(".zip"))
    print(f"{(time.time() - t0) / 3600:.1f} h  finished: {done}", flush=True)
for arm, p in procs.items():
    print(arm, "exit", p.returncode)
!ls -la /kaggle/working/
